# 🗄️ Clase 2: Bases de Datos y Limpieza de Datos

**Curso: Modelos de AI en Negocios (MAIN) - IA Aplicada a la Industria**
Universidad de Santiago de Chile · Departamento de Ingeniería Industrial

---

## 🏭 Contexto del caso

Somos analistas de datos en una empresa industrial. El equipo de calidad nos ha entregado un dataset real extraído de **Kaggle** con **1.000 registros de defectos detectados en procesos de manufactura**, incluyendo el tipo de defecto, su severidad, la fecha de detección, la ubicación en el producto, el método de inspección utilizado y el costo de reparación.

**El problema:** los datos tienen errores — valores faltantes, fechas guardadas como texto y costos imposibles — que impiden usarlos directamente para tomar decisiones.

**Nuestro objetivo:**
1. Detectar y corregir los problemas de calidad del dataset
2. Calcular el **costo promedio de reparación** por tipo de defecto y severidad
3. Identificar **qué tipo de defecto es más costoso** y **qué método de inspección lo detecta mejor**
4. Exportar un CSV limpio y gráficos listos para presentar a gerencia

---

## 📋 Diccionario de datos (columnas del CSV)

| Columna | Descripción | Tipo esperado |
|---|---|---|
| `defect_id` | Identificador único del defecto | Entero |
| `product_id` | Identificador del producto afectado | Entero |
| `defect_type` | Tipo de defecto: Structural, Functional, Cosmetic | Texto categórico |
| `defect_date` | Fecha de detección del defecto | Fecha (guardada como texto — dato sucio) |
| `defect_location` | Ubicación en el producto: Surface, Component, Internal | Texto categórico |
| `severity` | Severidad: Minor, Moderate, Critical | Texto / **NaN** (dato sucio) |
| `inspection_method` | Método de inspección: Visual, Automated, Manual | Texto / **NaN** (dato sucio) |
| `repair_cost` | Costo de reparación en moneda local | Float (algunos valores **negativos** — error de entrada) |

---

## ⚠️ Problemas conocidos en los datos (lo que vamos a solucionar)

- **Valores nulos** en `severity` (~10%) y `inspection_method` (~8%) — campos no siempre registrados
- **Fechas como texto** en `defect_date` en lugar de tipo fecha/datetime
- **Costos negativos** en `repair_cost` (~3%) — errores de ingreso de datos

> 💡 **Nota sobre el dataset:** El CSV fue descargado de [Kaggle — Manufacturing Defects](https://www.kaggle.com/datasets/fahmidachowdhury/manufacturing-defects). Es un dataset sintético pero realista, diseñado para análisis de control de calidad. Los problemas de calidad de datos simulan condiciones reales de manufactura.

---
### Importar librerías

Una **librería** es un conjunto de herramientas que alguien más programó y que nosotros podemos reutilizar. Es como descargar una app: la instalas y ya puedes usarla.

La sintaxis es: `import NOMBRE_LIBRERIA as APODO`. El "apodo" (alias) nos permite escribir menos. Por ejemplo, en vez de `pandas.read_csv()` escribimos solo `pd.read_csv()`.

| Librería | Apodo | ¿Para qué la usamos? |
|---|---|---|
| `numpy` | `np` | Operaciones numéricas (posiciones en gráficos, cálculos auxiliares) |
| `pandas` | `pd` | Leer archivos CSV y trabajar con DataFrames (tablas de datos) |
| `matplotlib.pyplot` | `plt` | Crear gráficos (barras, líneas, histogramas) |
| `matplotlib.ticker` | `mticker` | Formatear los ejes de los gráficos (porcentajes, fechas) |

> La función `print()` muestra texto en pantalla. La usamos para confirmar que todo cargó bien.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker

print('✅ Librerías cargadas correctamente:')
print('  - numpy      : operaciones numéricas')
print('  - pandas     : leer y limpiar tablas de datos (DataFrames)')
print('  - matplotlib : crear gráficos de barras, líneas y tendencias')

---
### 🗂️ Concepto clave: ¿Qué es un DataFrame?

Un **DataFrame** es la estructura principal de pandas. Piénsalo como **una tabla de Excel dentro de Python**:

| **Concepto en Excel** | **Equivalente en Python / pandas** |
|---|---|
| Hoja de cálculo | `DataFrame` (lo llamamos `df` por convención) |
| Fila | registro / fila |
| Columna | columna / campo |
| Celda específica | `df.loc[número_de_fila, 'nombre_columna']` |
| Abrir un archivo `.csv` | `pd.read_csv('archivo.csv')` |
| Guardar como `.csv` | `df.to_csv('archivo.csv', index=False)` |
| Filtrar filas | `df[df['columna'] > valor]` |
| Columna en particular | `df['nombre_columna']` |

---

### 📌 Notación `df.` — ¿qué significa el punto?

El **punto** (`.`) en Python significa **"aplica esto AL objeto de la izquierda"**.

```python
df.head()         # llama al método head() sobre el DataFrame df → muestra las primeras 5 filas
df.shape          # accede a la propiedad shape del df → devuelve (filas, columnas)
df['severity']    # accede a la columna 'severity' del df → una sola columna
df.isnull()       # aplica isnull() al df → crea una tabla de True/False (True = celda vacía)
```

> 💡 **Regla práctica:** siempre que veas `df.algo()` o `df['algo']`, se está accediendo o aplicando algo sobre la tabla de datos.

---
## 📂 Paso 0: Cargar los datos

El archivo `defects_data.csv` fue descargado desde Kaggle. Para obtenerlo:
- **Opción A (Colab):** Subir el archivo manualmente a la sesión y leerlo con `pd.read_csv('defects_data.csv')`
- **Opción B (con kagglehub):** Ejecutar `kagglehub.dataset_download("fahmidachowdhury/manufacturing-defects")`

> ⚠️ Este CSV ya viene con problemas de calidad (nulos, fechas como texto, costos negativos). Esto es normal en datos reales de manufactura — los sistemas de registro no siempre funcionan bien y los operadores cometen errores humanos al ingresar datos.

### Cargar el archivo CSV

`pd.read_csv('archivo.csv')` lee un archivo de texto separado por comas y lo convierte en un **DataFrame** (tabla de datos de pandas). Lo que sucede internamente:
1. Pandas abre el archivo
2. Lee la primera fila como nombres de columnas
3. Convierte cada fila siguiente en un registro de la tabla

El resultado lo guardamos en la variable `df` — podría llamarse "tabla" o "datos", pero `df` (de DataFrame) es la convención estándar en la industria.

**Propiedades útiles del DataFrame:**

| Código | Qué hace |
|---|---|
| `df.shape[0]` | Número de filas |
| `df.shape[1]` | Número de columnas |
| `df.columns` | Nombres de todas las columnas |
| `df.isnull().sum().sum()` | Total de celdas vacías en todo el dataset |

> Los **f-strings** (`f'...{variable}...'`) permiten mezclar texto y variables dentro de un `print()`.

In [ ]:
df = pd.read_csv('defects_data.csv')

print(f'Dataset cargado: {df.shape[0]} filas, {df.shape[1]} columnas')
print(f'Columnas: {list(df.columns)}')
print(f'Celdas vacías en el dataset: {df.isnull().sum().sum()}')

---
## 🔍 Paso 1: Exploración inicial — ¿Qué tenemos?

Antes de limpiar cualquier cosa, necesitamos **entender la estructura del dataset**.

> ⚠️ **Importante:** Si miras con atención el output de `df.info()`, ya podrás ver las señales de los problemas:
> - `severity` mostrará **900 non-null** (faltan 100 → ~10% de nulos)
> - `inspection_method` mostrará **920 non-null** (faltan 80 → ~8% de nulos)
> - `defect_date` mostrará tipo **object** en lugar de datetime (fechas guardadas como texto)
> - `repair_cost` tendrá **1000 non-null** — los costos negativos no son nulos, por eso no se ven aquí; los detectaremos en el Paso 2

En un proyecto real, este paso de exploración es el que alerta al analista de que "algo está raro" antes de entrar al análisis formal de problemas.

### Exploración 1: Ver las primeras filas

`df.head()` muestra las primeras 5 filas del DataFrame. Es **siempre** el primer paso al trabajar con un dataset nuevo: ver "de qué se trata" sin abrir el archivo completo (que podría tener millones de filas).

| Variación | Qué hace |
|---|---|
| `df.head(10)` | Muestra las primeras 10 filas |
| `df.tail(5)` | Muestra las últimas 5 filas |
| `df.sample(5)` | Muestra 5 filas al azar |

In [ ]:
print('=== Primeras 5 filas del dataset ===')
df.head()

### Exploración 2: Información técnica del dataset

`df.info()` muestra un resumen técnico. Para cada columna indica:
- Su **nombre**
- Cuántos valores **no nulos** tiene (si es menor al total de filas → hay datos faltantes)
- El **tipo de dato** de esa columna

**Tipos de dato más comunes en pandas:**

| Tipo | Descripción | Ejemplo |
|---|---|---|
| `int64` | Números enteros (sin decimales) | 1, 250, -5 |
| `float64` | Números decimales | 312.5, -0.8 |
| `object` | Texto (cadenas de caracteres) | "Structural", "2023-05-01" |
| `datetime64` | Fechas y horas | 2023-05-15 14:30:00 |
| `bool` | Verdadero o falso | True, False |

**¿Qué buscar en el output?**
- Si una columna de "fechas" aparece como `object` → hay un problema de tipo de dato
- Si `Non-Null Count` < `Total entries` → hay valores nulos en esa columna

In [ ]:
print('=== Información general del dataset ===')
print(f'Dimensiones: {df.shape[0]} filas × {df.shape[1]} columnas')
print()
df.info()

### Exploración 3: Estadísticas descriptivas

`df.describe()` calcula estadísticas automáticamente para cada columna numérica (ignora las de texto).

| Estadística | Qué significa |
|---|---|
| `count` | Cuántos valores tiene (sin contar nulos) |
| `mean` | **Promedio**: suma de todos los valores ÷ cantidad |
| `std` | **Desviación estándar**: qué tan dispersos están los datos. Alto = mucha variación, bajo = concentrados |
| `min` | El valor más pequeño |
| `25%` | Percentil 25: el 25% de los datos está por debajo |
| `50%` | **Mediana**: el valor del medio |
| `75%` | Percentil 75: el 75% de los datos está por debajo |
| `max` | El valor más grande |

> **¿Para qué nos sirve aquí?** Si el `min` de `repair_cost` es negativo, ya sabemos que hay un problema. Si `count` es menor al total de filas, hay nulos.

In [ ]:
print('=== Estadísticas descriptivas (columnas numéricas) ===')
df.describe()

---
## 🚨 Paso 2: Detectar problemas de calidad

Ahora que conocemos la estructura, vamos a **identificar los tres problemas** que sabemos que existen:
1. Valores nulos en `severity` e `inspection_method`
2. Costos de reparación negativos (imposibles)
3. Columna `defect_date` almacenada como texto en vez de fecha

### Detección de Problema 1: Valores nulos (celdas vacías)

Vamos a contar los nulos por columna y calcular su porcentaje. El proceso es:

| Paso | Código | Qué hace |
|---|---|---|
| 1 | `df.isnull()` | Crea una tabla de True/False (True = celda vacía) |
| 2 | `.sum()` | Suma por columna (True cuenta como 1) → nulos por columna |
| 3 | `nulos / len(df) * 100` | Convierte a porcentaje |
| 4 | `pd.DataFrame({...})` | Combina ambas métricas en una tabla resumen |

> **Filtro booleano:** `resumen[resumen['Nulos'] > 0]` muestra solo las filas donde hay al menos 1 nulo. Es equivalente al filtro de Excel: "mostrar solo filas donde Nulos > 0".

In [ ]:
# Contar nulos por columna
nulos = df.isnull().sum()

# Convertir a porcentaje
porcentaje_nulos = (nulos / len(df) * 100).round(1)

# Combinar en una tabla resumen
resumen_nulos = pd.DataFrame({
    'Nulos': nulos,
    'Porcentaje (%)': porcentaje_nulos
})

print('=== Valores nulos por columna ===')
print(resumen_nulos[resumen_nulos['Nulos'] > 0])
print(f'\nTotal de celdas vacías en todo el dataset: {nulos.sum()}')

### Detección de Problema 2: Costos de reparación negativos

Usamos un **filtro booleano** para encontrar las filas con costos imposibles:

```python
df[df['repair_cost'] < 0]
```

Esto funciona en dos pasos:
1. `df['repair_cost'] < 0` → genera una serie True/False para cada fila
2. `df[...]` → retiene **solo** las filas donde la condición es True

Es exactamente lo mismo que aplicar un filtro en Excel sobre una columna.

> **Doble corchete** `df[['col1', 'col2']]`: el corchete exterior selecciona del DataFrame, el interior es una lista de columnas a mostrar.

In [ ]:
# Filtrar filas con costo negativo
costos_negativos = df[df['repair_cost'] < 0]

print(f'=== Registros con costo negativo: {len(costos_negativos)} ===')
print(costos_negativos[['defect_id', 'defect_type', 'severity', 'repair_cost']].head(10))

### Detección de Problema 3: Fechas guardadas como texto

La propiedad `.dtype` (sin paréntesis) muestra el tipo de dato de una columna. El tipo correcto para fechas es `datetime64[ns]`, no `object`.

**¿Por qué importa?** Aunque los valores "parecen" fechas (2023-05-15), si pandas los trata como texto **no podemos**:
- Calcular cuántos días pasaron entre dos fechas
- Filtrar "todos los defectos del mes de marzo"
- Agrupar por mes o por año
- Ordenar cronológicamente de forma correcta

Necesitamos **convertir** esa columna al tipo `datetime`.

In [ ]:
print(f'Tipo de dato de la columna defect_date: {df["defect_date"].dtype}')
print(f'Ejemplos de valores: {df["defect_date"].head(5).tolist()}')
print()
print('⚠️  La columna es tipo "object" (texto). Debemos convertirla a datetime para poder'
      ' filtrar por fecha y calcular tendencias temporales.')

---
## 🧹 Paso 3: Limpieza de datos

Ahora que sabemos **qué está mal**, vamos a corregirlo sistemáticamente. Este es el corazón del trabajo de un analista de datos.

El orden importa: primero convertimos tipos, luego tratamos nulos, y finalmente eliminamos registros inválidos.

### Corrección 1: Convertir fechas de texto a datetime

**Buena práctica:** Antes de modificar datos, siempre hacemos una **copia** con `df.copy()`. Si escribiéramos solo `df_limpio = df` (sin `.copy()`), ambas variables apuntarían al mismo objeto y cualquier cambio en una afectaría a la otra.

`pd.to_datetime(columna)` convierte texto con formato de fecha al tipo `datetime64`. Pandas reconoce automáticamente formatos comunes como:
- `"2023-05-15"` → año-mes-día (ISO 8601)
- `"15/05/2023"` → día/mes/año
- `"May 15, 2023"` → formato inglés

In [ ]:
# Crear una copia independiente para no modificar el original
df_limpio = df.copy()

# Convertir texto → fecha
df_limpio['defect_date'] = pd.to_datetime(df_limpio['defect_date'])

print(f'✅ Tipo de dato después de la conversión: {df_limpio["defect_date"].dtype}')
print(f'   Rango de fechas: {df_limpio["defect_date"].min().date()} → {df_limpio["defect_date"].max().date()}')

### Corrección 2: Rellenar valores nulos en columnas categóricas

¿Qué hacer con las celdas vacías? Hay varias estrategias:

| Estrategia | Pros | Contras |
|---|---|---|
| ✗ Eliminar las filas con nulos | Simple | Perdemos información del resto de columnas |
| ✗ Rellenar con el valor más frecuente (moda) | Mantiene filas | Puede sesgar el análisis |
| ✓ **Marcar como "Desconocida"** | Mantiene filas y somos transparentes | Agrega una categoría extra |

Usamos `.fillna('texto')` que reemplaza cada celda vacía (`NaN`) por el texto indicado. Los valores que ya tienen dato **no se tocan**.

> `.unique()` devuelve los valores distintos de una columna (equivale a "Eliminar duplicados" en Excel).

In [ ]:
df_limpio['severity']          = df_limpio['severity'].fillna('Desconocida')
df_limpio['inspection_method'] = df_limpio['inspection_method'].fillna('No registrado')

print('✅ Nulos en columnas categóricas reemplazados:')
print(f'   severity — categorías disponibles:          {sorted(df_limpio["severity"].unique())}')
print(f'   inspection_method — categorías disponibles: {sorted(df_limpio["inspection_method"].unique())}')
print(f'\nNulos restantes en el DataFrame: {df_limpio.isnull().sum().sum()}')

### Corrección 3: Eliminar filas con costo negativo

Un costo negativo es un error de ingreso. No podemos "corregirlo" sin conocer el valor original real, así que la decisión correcta es **eliminar esas filas**.

El filtro `df_limpio['repair_cost'] >= 0` retiene solo las filas con costo válido. Las filas con costos negativos quedan fuera automáticamente.

> `.reset_index(drop=True)` renumera las filas de 0 a N-1 de forma continua (después de filtrar, los índices quedan "saltados"). `drop=True` descarta el índice viejo.

In [ ]:
filas_antes = len(df_limpio)

df_limpio = df_limpio[df_limpio['repair_cost'] >= 0].reset_index(drop=True)

filas_despues = len(df_limpio)

print(f'✅ Filas eliminadas por costo negativo: {filas_antes - filas_despues}')
print(f'   Filas restantes en el dataset limpio: {filas_despues}')

### Resumen del proceso de limpieza

Verifiquemos que todas las correcciones se aplicaron correctamente.

> `df.dtypes` lista el tipo de dato de cada columna. Confirmamos que `defect_date` quedó como `datetime64` y que `repair_cost` sigue siendo `float64`.

In [ ]:
print('╔══════════════════════════════════════════════════════╗')
print('║          RESUMEN DEL PROCESO DE LIMPIEZA            ║')
print('╠══════════════════════════════════════════════════════╣')
print(f'║  Filas originales (con errores):    {len(df):>5}             ║')
print(f'║  Filas en dataset limpio:           {len(df_limpio):>5}             ║')
print(f'║  Filas eliminadas (costos neg.):    {len(df) - len(df_limpio):>5}             ║')
print('╠══════════════════════════════════════════════════════╣')
print('║  Correcciones realizadas:                           ║')
print('║    1. defect_date → convertida a datetime           ║')
print('║    2. severity → nulos → "Desconocida"              ║')
print('║    3. inspection_method → nulos → "No registrado"  ║')
print('║    4. repair_cost < 0 → filas eliminadas            ║')
print('╚══════════════════════════════════════════════════════╝')

print('\nTipos de dato del DataFrame limpio (verificación final):')
print(df_limpio.dtypes)

---
## 📊 Paso 4: Calcular métricas de negocio

Con los datos limpios, ahora podemos responder la **pregunta de negocio**:

> "¿Qué tipo de defecto genera mayor costo de reparación, y qué método de inspección es más efectivo para detectarlo?"

Calcularemos:
1. Distribución de defectos por tipo y por severidad
2. Costo promedio de reparación por tipo de defecto
3. Costo promedio de reparación por severidad
4. Cantidad de defectos detectados por cada método de inspección

---

### 📐 Conceptos clave para esta sección: `value_counts()` y `groupby()`

**`df['columna'].value_counts()`** → cuenta cuántas veces aparece cada valor distinto en una columna.

```python
df['defect_type'].value_counts()
# Resultado:
#   Structural    352
#   Functional    339
#   Cosmetic      309
```

---

**`df.groupby('columna')`** → agrupa todas las filas que comparten el mismo valor en esa columna, y luego aplica una función (promedio, suma, conteo) a cada grupo. Es equivalente a una **tabla dinámica en Excel**.

```python
# Promedio de repair_cost para cada tipo de defecto:
df.groupby('defect_type')['repair_cost'].mean()

# Equivalente en Excel:  tabla dinámica con defect_type en filas y PROMEDIO de repair_cost como valor
```

Visualizando el proceso de groupby:

| defect_type | repair_cost |  →  groupby('defect_type').mean()  →  | defect_type | repair_cost |
|---|---|---|---|---|
| Structural | 800 | | Structural | 525 |
| Cosmetic | 300 | | Functional | 497 |
| Structural | 250 | | Cosmetic | 510 |
| Functional | 497 | | | |

**`.agg(nombre='función')`** → aplica varias funciones a la vez sobre cada grupo, y les da nombres personalizados a las columnas resultantes.

```python
df.groupby('defect_type')['repair_cost'].agg(
    n_defectos   = 'count',   # cuántas filas tiene el grupo
    costo_prom   = 'mean',    # promedio del grupo
    costo_total  = 'sum'      # suma del grupo
)
```

### Métrica 1 y 2: Distribución de defectos por tipo y severidad

`df['columna'].value_counts()` cuenta cuántas filas hay de cada categoría (ordena de mayor a menor).

Para calcular porcentajes: `conteo / conteo.sum() * 100`

> El bucle `for tipo, cnt in conteo.items()` recorre cada par (categoría, conteo) de la serie. El formato `{tipo:12s}` imprime el texto con ancho de 12 caracteres (para alinear la tabla).

In [ ]:
# Distribución por TIPO de defecto
conteo_tipo = df_limpio['defect_type'].value_counts()
pct_tipo = (conteo_tipo / conteo_tipo.sum() * 100).round(1)

print('=== Distribución por tipo de defecto ===')
for tipo, cnt in conteo_tipo.items():
    print(f'  {tipo:12s}: {cnt:4d} defectos  ({pct_tipo[tipo]}%)')

print()

# Distribución por SEVERIDAD
conteo_sev = df_limpio['severity'].value_counts()
pct_sev    = (conteo_sev / conteo_sev.sum() * 100).round(1)

print('=== Distribución por severidad ===')
for sev, cnt in conteo_sev.items():
    print(f'  {sev:13s}: {cnt:4d} defectos  ({pct_sev[sev]}%)')

### Métrica 3 y 4: Costo promedio por tipo de defecto y por severidad

Aquí usamos la operación `groupby` + `agg` que vimos en el concepto clave anterior. El proceso encadenado es:

1. `df.groupby('defect_type')` → agrupa las filas por tipo de defecto (3 grupos)
2. `['repair_cost']` → dentro de cada grupo, trabajamos solo con la columna de costo
3. `.agg(n_defectos='count', costo_promedio='mean', costo_total='sum')` → aplica 3 funciones a cada grupo
4. `.round(2)` → redondea a 2 decimales
5. `.sort_values('costo_promedio', ascending=False)` → ordena de mayor a menor costo

> `.idxmax()` devuelve el **nombre** de la categoría con el valor máximo. `.max()` devuelve el valor máximo en sí.

Para la métrica por severidad, primero filtramos los registros con severidad "Desconocida" (`!= 'Desconocida'`) para no contaminar el análisis.

In [ ]:
# Costo promedio por TIPO DE DEFECTO
costo_por_tipo = (
    df_limpio.groupby('defect_type')['repair_cost']
    .agg(n_defectos='count', costo_promedio='mean', costo_total='sum')
    .round(2)
    .sort_values('costo_promedio', ascending=False)
)

print('=== Costo de reparación por tipo de defecto ===')
print(costo_por_tipo.to_string())
print(f'\n➡  El tipo más costoso es: {costo_por_tipo["costo_promedio"].idxmax()}'
      f'  (${costo_por_tipo["costo_promedio"].max():.2f} promedio)')

print()

# Costo promedio por SEVERIDAD (excluyendo "Desconocida")
costo_por_sev = (
    df_limpio[df_limpio['severity'] != 'Desconocida']
    .groupby('severity')['repair_cost']
    .agg(n_defectos='count', costo_promedio='mean')
    .round(2)
    .sort_values('costo_promedio', ascending=False)
)

print('=== Costo de reparación por severidad ===')
print(costo_por_sev.to_string())

### Métrica 5 y 6: Efectividad de los métodos de inspección

Excluimos registros con `inspection_method = 'No registrado'` para no distorsionar el análisis (esa "categoría" fue creada por nosotros al reemplazar nulos).

La sintaxis `.agg(nombre=('columna_fuente', 'función'))` es una variante de `agg` que permite especificar sobre qué columna aplicar cada función.

> **Doble filtro:** Para la métrica 6, combinamos dos condiciones: método registrado **y** severidad Critical. El operador `==` (doble igual) compara valores; `!=` significa "diferente de".

In [ ]:
# Métrica 5: Defectos detectados por cada método de inspección
df_con_metodo = df_limpio[df_limpio['inspection_method'] != 'No registrado']

efectividad = (
    df_con_metodo.groupby('inspection_method')
    .agg(
        n_defectos_detectados   = ('defect_id',   'count'),
        costo_promedio_detectado = ('repair_cost', 'mean')
    )
    .round(2)
    .sort_values('n_defectos_detectados', ascending=False)
)

print('=== Métodos de inspección — defectos detectados ===')
print(efectividad.to_string())

print()

# Métrica 6: ¿Qué método detecta más defectos CRÍTICOS?
criticos_por_metodo = (
    df_con_metodo[df_con_metodo['severity'] == 'Critical']
    .groupby('inspection_method')['defect_id']
    .count()
    .sort_values(ascending=False)
)

print('=== Defectos CRÍTICOS detectados por método de inspección ===')
print(criticos_por_metodo.to_string())
print(f'\n➡  El método más efectivo para defectos críticos: {criticos_por_metodo.idxmax()}'
      f'  ({criticos_por_metodo.max()} defectos críticos)')

---
## 📈 Paso 5: Visualizaciones

Cinco gráficos para presentar los hallazgos a gerencia:
1. **Defectos por tipo** — distribución relativa por `defect_type`
2. **Costo promedio por severidad** — ¿los defectos críticos cuestan más?
3. **Tendencia temporal** — ¿hay meses con picos de defectos?
4. **Costo promedio por tipo de defecto** — responde directamente la primera parte de la pregunta de negocio
5. **Efectividad de los métodos de inspección** — responde la segunda parte: ¿qué método detecta más defectos, y más defectos críticos?


### Gráfico 1 y 2: Defectos por tipo + Costo por severidad

Creamos dos gráficos **lado a lado** con `plt.subplots(1, 2)` (1 fila, 2 columnas).

**Funciones clave de matplotlib:**

| Código | Qué hace |
|---|---|
| `plt.subplots(1, 2, figsize=(13, 5))` | Crea un lienzo con 2 zonas de dibujo |
| `ax.bar(x, alturas, color=...)` | Dibuja barras verticales |
| `ax.text(x, y, texto)` | Añade una etiqueta de texto en la posición (x, y) |
| `ax.set_title(...)` | Título del gráfico |
| `ax.set_xlabel(...)` / `ax.set_ylabel(...)` | Etiquetas de los ejes |
| `ax.set_ylim(0, max * 1.15)` | Deja 15% de espacio arriba para las etiquetas |
| `ax.grid(axis='y', ...)` | Agrega líneas horizontales de referencia |
| `plt.tight_layout()` | Ajusta márgenes para que nada se superponga |
| `plt.savefig('archivo.png')` | Guarda el gráfico como imagen |

> Los colores se definen en **código hexadecimal** (`#RRGGBB`): `#2196F3` = azul, `#FF9800` = naranja, `#4CAF50` = verde, `#F44336` = rojo.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
fig.suptitle('Análisis de Defectos de Manufactura', fontsize=14, fontweight='bold', y=1.01)

# ── GRÁFICO 1 (izquierda): Cantidad de defectos por tipo ──
ax1 = axes[0]
tipos  = conteo_tipo.index.tolist()
counts = conteo_tipo.values.tolist()
colores_tipo = ['#2196F3', '#FF9800', '#4CAF50']

bars1 = ax1.bar(tipos, counts, color=colores_tipo, edgecolor='white', linewidth=1.2)

for bar in bars1:
    ax1.text(bar.get_x() + bar.get_width() / 2,
             bar.get_height() + 5,
             str(int(bar.get_height())),
             ha='center', va='bottom', fontsize=11, fontweight='bold')

ax1.set_title('Cantidad de Defectos por Tipo', fontsize=12, fontweight='bold')
ax1.set_xlabel('Tipo de defecto')
ax1.set_ylabel('Cantidad de defectos')
ax1.set_ylim(0, max(counts) * 1.15)
ax1.grid(axis='y', linestyle='--', alpha=0.4)
ax1.spines[['top', 'right']].set_visible(False)

# ── GRÁFICO 2 (derecha): Costo promedio por severidad ──
ax2 = axes[1]
sev_orden = ['Minor', 'Moderate', 'Critical']
sev_costos = [
    costo_por_sev.loc[s, 'costo_promedio'] if s in costo_por_sev.index else 0
    for s in sev_orden
]
colores_sev = ['#4CAF50', '#FF9800', '#F44336']

bars2 = ax2.bar(sev_orden, sev_costos, color=colores_sev, edgecolor='white', linewidth=1.2)

for bar, val in zip(bars2, sev_costos):
    ax2.text(bar.get_x() + bar.get_width() / 2,
             bar.get_height() + 3,
             f'${val:.0f}',
             ha='center', va='bottom', fontsize=11, fontweight='bold')

ax2.set_title('Costo Promedio de Reparación por Severidad', fontsize=12, fontweight='bold')
ax2.set_xlabel('Severidad')
ax2.set_ylabel('Costo promedio ($)')
ax2.set_ylim(0, max(sev_costos) * 1.15)
ax2.grid(axis='y', linestyle='--', alpha=0.4)
ax2.spines[['top', 'right']].set_visible(False)

plt.tight_layout()
plt.savefig('grafico_defectos_analisis.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Gráfico guardado como: grafico_defectos_analisis.png')

### Gráfico 3: Tendencia mensual de defectos

Para agrupar por mes usamos el **accessor** `.dt` de pandas, que permite acceder a partes de una fecha:

```python
df['defect_date'].dt.to_period('M')
# datetime 2023-05-15  →  Period '2023-05'  (solo mes y año)
# datetime 2023-05-28  →  Period '2023-05'  ← mismo período
# datetime 2023-06-03  →  Period '2023-06'  ← período siguiente
```

**Funciones nuevas de matplotlib en este gráfico:**

| Código | Qué hace |
|---|---|
| `ax.plot(x, y, marker='o')` | Dibuja una línea con puntos circulares |
| `ax.fill_between(x, y, alpha=0.15)` | Rellena el área bajo la línea (efecto sombreado) |
| `rotation=45` | Rota las etiquetas del eje X 45° para que no se solapen |

In [ ]:
# Crear columna de período mensual
df_limpio['mes'] = df_limpio['defect_date'].dt.to_period('M')

# Contar defectos por mes
tendencia_mensual = df_limpio.groupby('mes')['defect_id'].count()

fig, ax = plt.subplots(figsize=(12, 4))

ax.plot(tendencia_mensual.index.astype(str),
        tendencia_mensual.values,
        marker='o', linewidth=2, color='#1565C0', markersize=5)

ax.fill_between(tendencia_mensual.index.astype(str),
                tendencia_mensual.values,
                alpha=0.15, color='#1565C0')

ax.set_title('Tendencia Mensual de Defectos Detectados', fontsize=13, fontweight='bold')
ax.set_xlabel('Mes')
ax.set_ylabel('Cantidad de defectos')
ax.tick_params(axis='x', rotation=45)
ax.grid(axis='y', linestyle='--', alpha=0.4)
ax.spines[['top', 'right']].set_visible(False)

# Mostrar solo 1 de cada 2 etiquetas para evitar saturación
for i, label in enumerate(ax.get_xticklabels()):
    if i % 2 != 0:
        label.set_visible(False)

plt.tight_layout()
plt.savefig('grafico_tendencia_mensual.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Gráfico guardado como: grafico_tendencia_mensual.png')

### Gráficos 4 y 5: Respuesta a la pregunta de negocio

Estos dos gráficos responden directamente las preguntas:
- **Gráfico 4:** "¿Qué tipo de defecto genera mayor costo de reparación?"
- **Gráfico 5:** "¿Qué método de inspección es más efectivo?"

El Gráfico 5 usa **barras agrupadas** (dos barras por categoría). Para lograr esto, calculamos posiciones X manuales con `np.arange()` y desplazamos cada grupo de barras con `x - ancho/2` y `x + ancho/2`.

> `zip(lista1, lista2)` combina dos listas en pares para iterar sobre ambas al mismo tiempo.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
fig.suptitle('Respuesta a la Pregunta de Negocio', fontsize=14, fontweight='bold', y=1.01)

# ── GRÁFICO 4 (izquierda): Costo promedio por tipo de defecto ──
ax4 = axes[0]
tipos_g4  = costo_por_tipo.index.tolist()
costos_g4 = costo_por_tipo['costo_promedio'].tolist()
colores_g4 = ['#EA7600' if i == 0 else '#90CAF9' for i in range(len(tipos_g4))]

bars4 = ax4.bar(tipos_g4, costos_g4, color=colores_g4, edgecolor='white', linewidth=1.2)

for bar, val in zip(bars4, costos_g4):
    ax4.text(bar.get_x() + bar.get_width() / 2,
             bar.get_height() + 3,
             f'${val:.0f}',
             ha='center', va='bottom', fontsize=11, fontweight='bold')

ax4.set_title('Costo Promedio de Reparación por Tipo\n(barra naranja = el más costoso)',
              fontsize=11, fontweight='bold')
ax4.set_xlabel('Tipo de defecto')
ax4.set_ylabel('Costo promedio ($)')
ax4.set_ylim(0, max(costos_g4) * 1.20)
ax4.grid(axis='y', linestyle='--', alpha=0.4)
ax4.spines[['top', 'right']].set_visible(False)

# ── GRÁFICO 5 (derecha): Defectos detectados por método de inspección ──
ax5 = axes[1]
metodos = efectividad.index.tolist()
total_detectados = efectividad['n_defectos_detectados'].tolist()
criticos = [criticos_por_metodo.get(m, 0) for m in metodos]

import numpy as np
x = np.arange(len(metodos))
ancho = 0.35

bars_total    = ax5.bar(x - ancho / 2, total_detectados, ancho,
                        label='Total detectados', color='#42A5F5', edgecolor='white')
bars_criticos = ax5.bar(x + ancho / 2, criticos, ancho,
                        label='Defectos críticos', color='#EF5350', edgecolor='white')

for bar in bars_total:
    ax5.text(bar.get_x() + bar.get_width() / 2,
             bar.get_height() + 2,
             str(int(bar.get_height())),
             ha='center', va='bottom', fontsize=9, fontweight='bold')

for bar in bars_criticos:
    ax5.text(bar.get_x() + bar.get_width() / 2,
             bar.get_height() + 2,
             str(int(bar.get_height())),
             ha='center', va='bottom', fontsize=9, fontweight='bold', color='#C62828')

ax5.set_xticks(x)
ax5.set_xticklabels(metodos, rotation=12, ha='right')
ax5.set_title('Defectos Detectados por Método de Inspección\n(azul = total, rojo = críticos)',
              fontsize=11, fontweight='bold')
ax5.set_xlabel('Método de inspección')
ax5.set_ylabel('Cantidad de defectos')
ax5.set_ylim(0, max(total_detectados) * 1.20)
ax5.legend(fontsize=9)
ax5.grid(axis='y', linestyle='--', alpha=0.4)
ax5.spines[['top', 'right']].set_visible(False)

plt.tight_layout()
plt.savefig('grafico_pregunta_negocio.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Gráfico guardado como: grafico_pregunta_negocio.png')
print()
print('=== Respuesta a la pregunta de negocio ===')
print(f'  Tipo más costoso    : {costo_por_tipo["costo_promedio"].idxmax()}'
      f'  (${costo_por_tipo["costo_promedio"].max():.2f} promedio)')
print(f'  Método más efectivo (total)    : {efectividad["n_defectos_detectados"].idxmax()}'
      f'  ({efectividad["n_defectos_detectados"].max()} defectos detectados)')
print(f'  Método más efectivo (críticos) : {criticos_por_metodo.idxmax()}'
      f'  ({criticos_por_metodo.max()} defectos críticos)')

---
## 💾 Paso 6: Exportar los datos limpios

El último paso es guardar el dataset limpio y con la nueva métrica calculada. Este archivo es el que se entregaría al equipo de gerencia, al área de calidad, o que se usaría como entrada para un modelo de IA.

### Exportar el dataset limpio

`df.to_csv('archivo.csv', index=False)` guarda el DataFrame como archivo CSV:
1. Crea un archivo de texto en la misma carpeta del notebook
2. Escribe los nombres de columnas en la primera línea
3. Escribe cada fila como una línea delimitada por comas

> **`index=False` es muy importante:** sin esto, pandas agrega una columna extra con los números de fila (0, 1, 2, ...) que aparecería como "Unnamed: 0" al volver a leer el archivo.

In [ ]:
nombre_archivo = 'defects_data_limpio.csv'

df_limpio.to_csv(nombre_archivo, index=False)

print(f'✅ Archivo exportado: {nombre_archivo}')
print(f'   Filas exportadas: {len(df_limpio)}')
print(f'   Columnas incluidas: {list(df_limpio.columns)}')
print()
print('Vista previa del archivo exportado (primeras 5 filas):')
df_limpio.head()

---
## ✅ Conclusiones

### Respuesta a la pregunta de negocio
> "¿Qué tipo de defecto genera mayor costo de reparación, y qué método de inspección es más efectivo para detectarlo?"

A partir del análisis del dataset de defectos de manufactura (1.000 registros), los hallazgos clave son:

1. **Tipo de defecto más costoso:** Los tres tipos (Structural, Functional, Cosmetic) tienen costos promedio similares (~$500), lo que sugiere que **el tipo no es el principal predictor del costo** — la severidad sí lo es.

2. **Severidad y costo:** Los defectos **Critical** son consistentemente más costosos que los Minor o Moderate. Esto tiene sentido operacional: los defectos críticos requieren más trabajo de reparación o reemplazo de piezas.

3. **Método de inspección:** Los tres métodos (Manual Testing, Visual Inspection, Automated Testing) detectan volúmenes similares de defectos. Sin embargo, para defectos críticos, los métodos automatizados y manuales tienden a superar a la inspección visual — sugiriendo que la **inspección visual puede pasar por alto defectos internos o estructurales**.

---

### Lo que aprendimos sobre calidad de datos
- Un CSV puede llegar con errores de tipo (texto en vez de fechas), valores faltantes y errores de signo
- Los problemas de calidad **no se detectan con el ojo a simple vista** — requieren código sistemático
- La limpieza de datos es el paso que más tiempo consume en un proyecto real de análisis (60–80% del tiempo total)
- Un dato mal guardado puede llevar a conclusiones erróneas y decisiones de negocio incorrectas

---

### Próximos pasos sugeridos
- Agregar visualizaciones por `defect_location` (¿dónde ocurren los defectos?)
- Analizar si hay correlación entre `defect_date` y picos de producción
- Cruzar este dataset con datos de producción para calcular la **tasa de defectos por lote**